# U-Net segmentation training on Google Colab

This notebook runs `002_segmentation/unet_holdout/train.py` using a Colab GPU. The dataset is copied from Google Drive to Colab's local storage for faster `.npy` file access, while logs and checkpoints are saved directly to Google Drive so they persist after the runtime ends.

Before running the notebook, select **Runtime > Change runtime type > T4 GPU**. Make sure the selected repository branch has been pushed to GitHub.

## 1. Check the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is unavailable. Enable it through Runtime > Change runtime type."
    )

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Configure Colab

Adjust all paths in this cell. `DRIVE_DATASET_ROOT` must point directly to the `_segmentation_dataset_v2` directory containing the metadata CSV, images, and masks. Use `RESUME_CHECKPOINT_PATH = None` for a new training run.

In [ ]:
from pathlib import Path

# repository settings
REPOSITORY_URL = (
    "https://github.com/FillipusAditya/mask-guided-lung-nodule-xai.git"
)
REPOSITORY_BRANCH = "refactor/002-segmentation"
PROJECT_ROOT = Path("/content/mask-guided-lung-nodule-xai")

# dataset settings
DRIVE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/lung-nodule/_segmentation_dataset_v2"
)
COPY_DATASET_TO_LOCAL = True
LOCAL_DATASET_ROOT = Path(
    "/content/segmentation_data/_segmentation_dataset_v2"
)

# persistent output settings
DRIVE_OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/lung-nodule/"
    "segmentation_results/unet_holdout_split"
)

# training settings
TOTAL_EPOCHS = 100
BATCH_SIZE = 2
EARLY_STOPPING_PATIENCE = 5
NUM_WORKERS = 2

# use None for a new run or an absolute Drive path for resume
RESUME_CHECKPOINT_PATH = None
# Example:
# RESUME_CHECKPOINT_PATH = (
#     "/content/drive/MyDrive/lung-nodule/segmentation_results/"
#     "unet_holdout_split/<uuid>/last_checkpoint.pth"
# )

## 4. Clone or update the repository

In [ ]:
import subprocess

if (PROJECT_ROOT / ".git").exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "fetch", "origin"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git",
            "-C",
            str(PROJECT_ROOT),
            "pull",
            "--ff-only",
            "origin",
            REPOSITORY_BRANCH,
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            REPOSITORY_BRANCH,
            "--single-branch",
            REPOSITORY_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )

print(f"Repository ready: {PROJECT_ROOT}")

## 5. Install dependencies

The Torch and Torchvision versions provided by Colab are preserved to maintain compatibility with the CUDA runtime. This cell installs only missing packages.

In [ ]:
import importlib.util
import sys

required_packages = {
    "albumentations": "albumentations",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]

if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )

if importlib.util.find_spec("torchvision") is None:
    raise RuntimeError("Torchvision is unavailable in the Colab runtime.")

print("Dependencies are ready.")

## 6. Prepare the dataset

When `COPY_DATASET_TO_LOCAL=True`, the dataset is copied to `/content`. A `.colab_copy_complete` marker is created after the process finishes. If copying is interrupted, rerunning this cell resumes synchronization. Set `COPY_DATASET_TO_LOCAL=False` only when reading the dataset directly from Drive.

In [ ]:
if not DRIVE_DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f"Drive dataset directory not found: {DRIVE_DATASET_ROOT}"
    )

if COPY_DATASET_TO_LOCAL:
    LOCAL_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    copy_marker = LOCAL_DATASET_ROOT / ".colab_copy_complete"

    if not copy_marker.exists():
        subprocess.run(
            [
                "rsync",
                "-a",
                "--info=progress2",
                f"{DRIVE_DATASET_ROOT}/",
                f"{LOCAL_DATASET_ROOT}/",
            ],
            check=True,
        )
        copy_marker.touch()

    DATASET_ROOT = LOCAL_DATASET_ROOT
else:
    DATASET_ROOT = DRIVE_DATASET_ROOT

metadata_path = DATASET_ROOT / "001_holdout_split_lidc_lndb.csv"
if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")

print(f"Dataset ready: {DATASET_ROOT}")
print(f"Metadata: {metadata_path}")

## 7. Write the Colab training configuration

This cell updates the configuration in the temporary Colab clone. It does not modify the configuration file in your local repository.

In [ ]:
import json

config_path = PROJECT_ROOT / "002_segmentation/configs/unet_holdout.json"

with config_path.open("r", encoding="utf-8") as file:
    config = json.load(file)

config["data"]["dataset_root"] = str(DATASET_ROOT)
config["output"]["root_directory"] = str(DRIVE_OUTPUT_ROOT)
config["training"]["total_epochs"] = TOTAL_EPOCHS
config["training"]["batch_size"] = BATCH_SIZE
config["training"]["early_stopping_patience"] = (
    EARLY_STOPPING_PATIENCE
)
config["dataloader"]["num_workers"] = NUM_WORKERS
config["checkpoint"]["resume_checkpoint_path"] = (
    str(RESUME_CHECKPOINT_PATH)
    if RESUME_CHECKPOINT_PATH is not None
    else None
)

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if RESUME_CHECKPOINT_PATH is not None:
    resume_path = Path(RESUME_CHECKPOINT_PATH)
    if not resume_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {resume_path}")
    if not (resume_path.parent / "training_log.csv").is_file():
        raise FileNotFoundError(
            f"Training log not found: {resume_path.parent}"
        )

with config_path.open("w", encoding="utf-8") as file:
    json.dump(config, file, indent=4)

print(json.dumps(config, indent=4))

## 8. Preflight the dataset

This cell loads one training sample and one validation sample to verify the CSV, `.npy` paths, transforms, and tile splitting before starting a long training run.

In [ ]:
import sys

import albumentations as A

segmentation_root = PROJECT_ROOT / "002_segmentation"
if str(segmentation_root) not in sys.path:
    sys.path.insert(0, str(segmentation_root))

from unet_utils.dataset import LungDataset

transforms = A.Compose(
    [
        A.Resize(
            height=config["data"]["input_height"],
            width=config["data"]["input_width"],
        )
    ]
)

dataset_arguments = {
    "root_dir": DATASET_ROOT,
    "split_method": config["data"]["split_method"],
    "metadata_filename": config["data"]["metadata_filename"],
    "fold": config["data"]["fold"],
    "image_path_column": config["data"]["image_path_column"],
    "tile_grid_size": config["data"]["tile_grid_size"],
    "transform": transforms,
}

train_dataset = LungDataset(split="train", **dataset_arguments)
val_dataset = LungDataset(split="val", **dataset_arguments)

if len(train_dataset) == 0 or len(val_dataset) == 0:
    raise RuntimeError("The training or validation dataset is empty.")

image_tiles, mask_tiles = train_dataset[0]

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Image tiles: {tuple(image_tiles.shape)} {image_tiles.dtype}")
print(f"Mask tiles: {tuple(mask_tiles.shape)} {mask_tiles.dtype}")
print(f"Mask values: {torch.unique(mask_tiles).tolist()}")

## 9. Run training

A checkpoint is saved at the end of every epoch. If this cell is interrupted during an epoch, training resumes from the most recent fully saved epoch.

In [ ]:
import os

training_script = (
    PROJECT_ROOT / "002_segmentation/unet_holdout/train.py"
)
training_environment = os.environ.copy()
training_environment["MPLCONFIGDIR"] = "/content/matplotlib"
training_environment["PYTHONUNBUFFERED"] = "1"

subprocess.run(
    [sys.executable, str(training_script)],
    cwd=PROJECT_ROOT,
    env=training_environment,
    check=True,
)

## 10. Inspect the output and prepare the resume path

In [ ]:
import pandas as pd
from IPython.display import Image, display

if RESUME_CHECKPOINT_PATH is not None:
    latest_output_dir = Path(RESUME_CHECKPOINT_PATH).parent
else:
    completed_runs = [
        directory
        for directory in DRIVE_OUTPUT_ROOT.iterdir()
        if directory.is_dir()
        and (directory / "last_checkpoint.pth").is_file()
    ]
    if not completed_runs:
        raise FileNotFoundError("No new training output was found.")
    latest_output_dir = max(
        completed_runs,
        key=lambda directory: (directory / "last_checkpoint.pth").stat().st_mtime,
    )

training_log_path = latest_output_dir / "training_log.csv"
last_checkpoint_path = latest_output_dir / "last_checkpoint.pth"

print(f"Output directory: {latest_output_dir}")
print(f"Resume checkpoint: {last_checkpoint_path}")
display(pd.read_csv(training_log_path).tail())

visualization_paths = [
    latest_output_dir / "visualizations/loss_curve.png",
    latest_output_dir / "visualizations/dice_curve.png",
    latest_output_dir / "visualizations/iou_curve.png",
    latest_output_dir / "visualizations/metrics_curve.png",
]

for visualization_path in visualization_paths:
    if visualization_path.is_file():
        print(visualization_path.name)
        display(Image(filename=str(visualization_path), width=600))

print("Use the following value in the next session:")
print(f'RESUME_CHECKPOINT_PATH = r"{last_checkpoint_path}"')

## Resume in the next Colab session

1. Rerun the GPU, Drive mount, configuration, clone, dependency, and dataset cells.
2. In the configuration cell, set `RESUME_CHECKPOINT_PATH` to the path printed by the final cell.
3. Set `TOTAL_EPOCHS` to the total target, not the number of additional epochs. For example, a checkpoint at epoch 20 with `TOTAL_EPOCHS = 100` resumes at epoch 21 and continues through epoch 100.
4. Rerun the configuration-writing, preflight, and training cells.

If early stopping has already reached a patience of 5 in the checkpoint, the program will not continue training.